<hr style=\"color:#5A7D9F;\">
<p align=\"center\">
    <b style=\"font-size:2vw; color:#5A7D9F; font-weight:bold;\">
    <center>Benthic Traits Mapping</center>
    </b>
</p>
<hr style=\"color:#5A7D9F;\">

In [5]:
# ---------- Librairies ----------
import matplotlib.pyplot as plt
import xarray as xr

# ---------- Jupyter ----------
%matplotlib inline
plt.rcParams.update({"font.size": 13})

# Making sure modules are reloaded when modified
%reload_ext autoreload
%autoreload 2

<hr style=\"color:#5A7D9F;\">
<p align=\"center\">
    <b style=\"font-size:2vw; color:#5A7D9F; font-weight:bold;\">
    <center>Playground</center>
    </b>
</p>
<hr style=\"color:#5A7D9F;\">

In [6]:
from benthic.config import PATH_DATASET_TRAIN_VAL
from benthic.data.const import DATASET_INPUT_VARIABLES, DATASET_OUTPUT_VARIABLES
from benthic.data.dataset import get_datasets
from benthic.data.tools import Statistics, from_zarr_to_numpy

# Loading the dataset
ds = xr.open_zarr(PATH_DATASET_TRAIN_VAL)

# Extracting the input and output variables
X, y = (
    from_zarr_to_numpy(ds, DATASET_INPUT_VARIABLES),
    from_zarr_to_numpy(ds, DATASET_OUTPUT_VARIABLES),
)
print(f"Input shape: {X.shape}", f" | Output shape: {y.shape}")

# Standardizing data
stats_X, stats_y = Statistics(), Statistics()

stats_X.fit(X)
stats_y.fit(y)

X_standardized = stats_X.standardize(X)
y_standardized = stats_y.standardize(y)

# Create nested k-fold splits using standardized data
X_train, y_train, X_val, y_val, X_test, y_test, coords_train, coords_val, coords_test = (
    get_datasets(
        X_standardized,
        y_standardized,
        r_validation=2,
        k_validation=4,
        k_test=5,
        seed_validation=42,
        seed_test=42,
    )
)

# Displaying shapes
print(
    "Training Folds | ",
    len(X_train),
    "folds with shapes:",
    [X_train[i].shape for i in range(len(X_train))],
)
print(
    "Validation Folds | ",
    len(X_val),
    "folds with shapes:",
    [X_val[i].shape for i in range(len(X_val))],
)
print(
    "Test Folds | ",
    len(X_test),
    "folds with shapes:",
    [X_test[i].shape for i in range(len(X_test))],
)